# Práctica guiada · Sesión 7

**Versión con código, para quien programa.** Reproduce los mismos números que la hoja `Laboratorio_S07.xlsx`: si algo no coincide, algo se calculó mal.

Corre tal cual en Google Colab o en Replit. Solo usa `pandas`, que ya viene instalado. El hilo del día es el mismo del pizarrón: la IA (o el código) entrega un candidato, y el número es correcto solo cuando revisas que el cálculo hizo lo que pediste y que los datos lo sostienen.


## El padrón (ficticio)

Lo cargamos tal como llega un registro administrativo real: todo como texto. Ahí empiezan los problemas, porque los montos ampliados vienen con coma de miles y un promedio ingenuo no los reconoce como número.


In [1]:
import pandas as pd
import numpy as np

# (folio, entidad, monto_apoyo tal como se capturo, estatus)
registro = [
    ("F-001", "Norte",  "2000",  "Activo"),
    ("F-002", "Centro", "5,000", "activo"),
    ("F-003", "Sur",    "2000",  "ACTIVO"),
    ("F-004", "Norte",  "5,000", "Activo"),
    ("F-005", "Centro", "2000",  "Baja"),
    ("F-006", "Sur",    "2000",  "Activo "),
    ("F-007", "Norte",  "5,000", "Activo"),
    ("F-008", "Centro", "2000",  "baja"),
    ("F-009", "Sur",    "2000",  "Activo"),
    ("F-010", "Norte",  "5,000", "Activo"),
    ("F-011", "Centro", "2000",  "Activo"),
    ("F-012", "Sur",    "5,000", "Activo"),
    ("F-013", "Norte",  "2000",  "Activo"),
    ("F-014", "Centro", "5,000", "Activo"),
    ("F-015", "Sur",    "2000",  "Baja"),
    ("F-003", "Sur",    "2000",  "ACTIVO"),   # folio duplicado
    ("F-009", "Sur",    "2000",  "Activo"),   # folio duplicado
    ("F-016", "Norte",  "",      "Activo"),   # monto vacio
]
df = pd.DataFrame(registro, columns=["folio", "entidad", "monto_apoyo", "estatus"])
print(df.shape, "renglones x columnas")
df.head()


(18, 4) renglones x columnas


,folio,entidad,monto_apoyo,estatus
0,F-001,Norte,2000,Activo
1,F-002,Centro,"5,000",activo
2,F-003,Sur,2000,ACTIVO
3,F-004,Norte,"5,000",Activo
4,F-005,Centro,2000,Baja


## Parte 3 primero: EDA (mirar antes de calcular)

Antes de cualquier estadístico, el barrido de exploración. `pandas` lo hace en pocas líneas, y aquí sí revela lo que Excel esconde: `nunique()` distingue mayúsculas, así que ve las cuatro formas de escribir «Activo».


In [2]:
# Tipos y faltantes
print("Tipos de columna:")
print(df.dtypes, "\n")

# monto como numero, limpiando la coma de miles
monto_limpio = pd.to_numeric(df["monto_apoyo"].str.replace(",", "", regex=False), errors="coerce")

print("Faltantes de monto:", monto_limpio.isna().sum())
print("Monto minimo / maximo:", monto_limpio.min(), "/", monto_limpio.max())
print("Folios duplicados:", int(df["folio"].duplicated().sum()))
print()
print("Estatus, formas distintas (sin limpiar):", df["estatus"].nunique())
print("  ->", list(df["estatus"].unique()))
print("Estatus, ya limpio:", df["estatus"].str.strip().str.capitalize().nunique(),
      "->", list(df["estatus"].str.strip().str.capitalize().unique()))


Tipos de columna:
folio          str
entidad        str
monto_apoyo    str
estatus        str
dtype: object 

Faltantes de monto: 1
Monto minimo / maximo: 2000.0 / 5000.0
Folios duplicados: 2

Estatus, formas distintas (sin limpiar): 6
  -> ['Activo', 'activo', 'ACTIVO', 'Baja', 'Activo ', 'baja']
Estatus, ya limpio: 2 -> ['Activo', 'Baja']


## Parte 2: el promedio ingenuo contra el corregido

Este bloque es lo que generaría un **prompt preciso**: «promedio de monto_apoyo por beneficiario, tratando el texto como número, ignorando vacías y contando cada folio una sola vez». Un prompt vago («saca el promedio del apoyo») habría producido `df["monto_apoyo"].mean()`, que ni siquiera corre porque la columna es texto.


In [3]:
# Ingenuo: convierto a numero pero NO limpio la coma ni quito duplicados.
# Las becas ampliadas ("5,000") no se reconocen como numero y se caen.
monto_ingenuo = pd.to_numeric(df["monto_apoyo"], errors="coerce")
promedio_ingenuo = monto_ingenuo.mean()
print(f"Promedio ingenuo   = {promedio_ingenuo:,.0f}   (sobre {monto_ingenuo.notna().sum()} celdas)")

# Corregido: limpio la coma, quito folios duplicados y excluyo el monto vacio.
df["monto_limpio"] = pd.to_numeric(df["monto_apoyo"].str.replace(",", "", regex=False), errors="coerce")
validos = df.drop_duplicates(subset="folio", keep="first").dropna(subset=["monto_limpio"])
promedio_correcto = validos["monto_limpio"].mean()
print(f"Promedio corregido = {promedio_correcto:,.0f}   (sobre {len(validos)} beneficiarios unicos)")

brecha = (promedio_correcto - promedio_ingenuo) / promedio_correcto
print(f"El ingenuo subestima el apoyo en {brecha:.1%}")


Promedio ingenuo   = 2,000   (sobre 11 celdas)
Promedio corregido = 3,200   (sobre 15 beneficiarios unicos)
El ingenuo subestima el apoyo en 37.5%


El promedio ingenuo (2,000) subestima el apoyo en 37.5% frente al corregido (3,200). El código no mintió: promedió lo que pudo leer como número, y todas las becas ampliadas estaban guardadas como texto. El error vivía en los datos, no en la fórmula.

## Extra: el promedio por entidad (ya limpio)

Con los datos limpios, el desglose que pedía el prompt sale de una línea.


In [4]:
por_entidad = (validos
    .groupby("entidad")["monto_limpio"]
    .agg(promedio="mean", beneficiarios="count")
    .round(0))
print(por_entidad)


         promedio  beneficiarios
entidad                         
Centro     3200.0              5
Norte      3800.0              5
Sur        2600.0              5


## Cierre

Este notebook ya es lo que abre la Sesión 8: un flujo reproducible. Cualquiera puede volver a correrlo sobre el mismo padrón y obtener 2,000, 3,200 y el desglose por entidad, con la limpieza escrita paso a paso en vez de a mano. La bitácora del Excel es exactamente esto, pero en código: el registro de qué se limpió y por qué, que hace defendible cada número.
